# Example 3: Load dataset and run all available `plot_de_*` API functions

This notebook runs every currently exposed gene-level plotting function in `scomnom.plotting`.

Notes:
- Some plots require specific metadata keys (for example `groupby`, `sample_key`, UMAP embedding).
- The helper cells below infer sensible defaults and skip plots when requirements are missing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scomnom as om

# Edit this path
in_path = Path("/path/to/input_dataset.zarr")


In [ ]:
adata = om.load_dataset(in_path)
adata


In [ ]:
def pick_obs_key(adata, preferred):
    for k in preferred:
        if k in adata.obs:
            return k
    for k in adata.obs.columns:
        s = adata.obs[k]
        try:
            if str(s.dtype) == "category" or s.astype(str).nunique() <= 30:
                return k
        except Exception:
            pass
    return None


def ensure_umap(adata):
    if "X_umap" in adata.obsm:
        return
    if "X_pca" not in adata.obsm:
        sc.pp.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)


groupby_key = pick_obs_key(adata, ["ident", "leiden", "cluster", "celltype", "annotation"])
sample_key = pick_obs_key(adata, ["sample_id", "sample", "donor", "batch"])
condition_key = pick_obs_key(adata, ["condition", "group", "treatment", "status"])

if groupby_key is None:
    raise ValueError("Could not infer a grouping key in adata.obs for DE plotting.")

genes = [g for g in adata.var_names[:12].tolist() if isinstance(g, str)]
if len(genes) < 3:
    raise ValueError("Need at least a few genes in adata.var_names for plotting examples.")

# Synthetic DE table for volcano demonstration
df_de = pd.DataFrame({
    "gene": adata.var_names[: min(400, adata.n_vars)].astype(str),
})
rng = np.random.default_rng(42)
df_de["log2FoldChange"] = rng.normal(loc=0.0, scale=1.8, size=len(df_de))
df_de["padj"] = np.clip(rng.uniform(1e-6, 1.0, size=len(df_de)), 1e-12, 1.0)

print(f"groupby_key: {groupby_key}")
print(f"sample_key: {sample_key}")
print(f"condition_key: {condition_key}")


In [ ]:
# 1) Volcano
om.plotting.plot_de_volcano(
    df_de,
    padj_thresh=0.05,
    lfc_thresh=1.0,
    top_label_n=12,
    display=True,
)


In [ ]:
# 2) Dotplot top genes
om.plotting.plot_de_dotplot_top_genes(
    adata,
    genes=genes,
    groupby=groupby_key,
    dot_min=0,
    dot_max=180,
    display=True,
)


In [ ]:
# 3) Heatmap top genes
om.plotting.plot_de_heatmap_top_genes(
    adata,
    genes=genes,
    groupby=groupby_key,
    display=True,
)


In [ ]:
# 4) Violin grid genes (set dot_size=0 to hide dots)
om.plotting.plot_de_violin_grid_genes(
    adata,
    genes=genes,
    groupby=groupby_key,
    stripplot=True,
    dot_size=0,
    display=True,
)


In [ ]:
# 5) Violin genes
om.plotting.plot_de_violin_genes(
    adata,
    genes=genes[:4],
    groupby=groupby_key,
    dot_size=2.0,
    display=True,
)


In [ ]:
# 6) UMAP feature grid
ensure_umap(adata)
om.plotting.plot_de_umap_features_grid(
    adata,
    genes=genes[:6],
    ncols=3,
    show_umap_corner_axes=True,
    display=True,
)


In [ ]:
# 7) Single UMAP (default ident-like key) + custom corner axes
om.plotting.plot_de_umap_single(
    adata,
    show_umap_corner_axes=True,
    display=True,
)


In [ ]:
# 8) Sample-level heatmap (requires sample_key)
if sample_key is not None:
    om.plotting.plot_de_heatmap_top_genes_by_sample(
        adata,
        genes=genes,
        sample_key=sample_key,
        condition_key=condition_key,
        display=True,
    )
else:
    print("Skipped plot_de_heatmap_top_genes_by_sample: no sample-like key found in adata.obs")
